In [2]:
import pandas as pd

df=pd.read_csv("RetailCleaned.csv",header=None)
df.columns=['Invoice','StockCode','Description','Quantity','InvoiceDate','Price','Customer_ID','Country','SourceSheet']

df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])

print(df.shape)
df.info()

C:\Users\LOQ\AppData\Local\Temp\ipykernel_18612\683656124.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("RetailCleaned.csv",header=None)


(1033036, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  object        
 2   Description  1028761 non-null  object        
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  object        
 8   SourceSheet  1033036 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 70.9+ MB


In [3]:
print(df.shape)
df.info()

(1033036, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  object        
 2   Description  1028761 non-null  object        
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  object        
 8   SourceSheet  1033036 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 70.9+ MB


In [4]:
print(df.head())

  Invoice StockCode                         Description  Quantity  \
0  489434     21232      STRAWBERRY CERAMIC TRINKET BOX        24   
1  489434     21523  FANCY FONT HOME SWEET HOME DOORMAT        10   
2  489434     21871                 SAVE THE PLANET MUG        24   
3  489434     22041        RECORD FRAME 7" SINGLE SIZE         48   
4  489434     22064          PINK DOUGHNUT TRINKET POT         24   

          InvoiceDate  Price  Customer_ID         Country SourceSheet  
0 2009-12-01 07:45:00   1.25      13085.0  United Kingdom   2010-2011  
1 2009-12-01 07:45:00   5.95      13085.0  United Kingdom   2010-2011  
2 2009-12-01 07:45:00   1.25      13085.0  United Kingdom   2010-2011  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom   2010-2011  
4 2009-12-01 07:45:00   1.65      13085.0  United Kingdom   2010-2011  


In [6]:
rfm_data=df[(df['Customer_ID'].notna()) & (df['Quantity']>0)].copy()
snapshot_date=pd.to_datetime("2011-12-10")

rfm=rfm_data.groupby('Customer_ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date-x.max()).days),
    Frequency=('Invoice','nunique'),
    Monetary=('Quantity', lambda x: (x*rfm_data.loc[x.index,'Price']).sum())
).reset_index()

In [7]:
# SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['Recency'] > 90

In [8]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT
df = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
    rfm[['Customer_ID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='Customer_ID', how='left'
)

df_final.to_excel('EcommerceDashboardData.xlsx', index=False)